# Streaming de Eventos de Registros de Memória com Amazon Bedrock AgentCore Memory

## Visão Geral

Este tutorial demonstra como configurar o [**streaming de registros de memória**](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-record-streaming.html) com o Amazon Bedrock AgentCore Memory. Você configurará um [Amazon Kinesis Data Stream](https://docs.aws.amazon.com/streams/latest/dev/introduction.html) para receber notificações em tempo real quando registros de memória são criados, atualizados ou excluídos — possibilitando arquiteturas orientadas a eventos sem a necessidade de polling em APIs.

### Detalhes do Tutorial

| Informação          | Detalhes                                                         |
|:--------------------|:-----------------------------------------------------------------|
| Tipo do tutorial    | Streaming de Registros de Memória                                |
| Funcionalidade      | Streaming de Eventos de Memória de Longo Prazo                   |
| Recursos principais | Kinesis Data Streams, Eventos do Ciclo de Vida de Registros de Memória |
| Complexidade        | Intermediário                                                    |
| SDK utilizado       | boto3                                                            |

### O Que Você Aprenderá

Neste tutorial, você aprenderá como:
1. Criar um Amazon Kinesis Data Stream para receber eventos de registros de memória
2. Configurar uma IAM role para o AgentCore Memory publicar no seu stream
3. Criar um recurso de memória com streaming habilitado
4. Disparar e consumir eventos do ciclo de vida de registros de memória em tempo real
5. Configurar níveis de conteúdo dos eventos (`FULL_CONTENT` ou `METADATA_ONLY`)
   
### Como Funciona

O streaming de registros de memória utiliza um modelo de entrega baseado em push. Quando registros de memória são alterados, eventos são automaticamente publicados no seu Kinesis Data Stream:

- **MemoryRecordCreated** — Disparado pela extração de memória de longo prazo ou pela API `BatchCreateMemoryRecords`
- **MemoryRecordUpdated** — Disparado pela API `BatchUpdateMemoryRecords`
- **MemoryRecordDeleted** — Disparado por workflows de consolidação, `DeleteMemoryRecord` ou pela API `BatchDeleteMemoryRecords`

### Arquitetura

<div style="text-align:left">
    <img src="memory_record_streaming.png" width="90%"/>
</div>

## 0. Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas com acesso ao AgentCore Memory, Kinesis e IAM
* Acesso a modelos do Amazon Bedrock (para extração de memória de longo prazo)

Primeiro, vamos instalar as bibliotecas necessárias:

In [ ]:
!pip install boto3>=1.42.63

### Configurando o Ambiente

Vamos importar as bibliotecas necessárias e configurar nosso ambiente:

In [ ]:
import os
import json
import time
import uuid
import base64
import logging
import boto3
from datetime import datetime, timezone
from botocore.exceptions import ClientError

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("memory-streaming")

REGION = os.getenv('AWS_REGION', 'us-west-2')
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

# Initialize boto3 clients
kinesis_client = boto3.client('kinesis', region_name=REGION)
iam_client = boto3.client('iam')
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)

# Unique identifier for resource naming
unique_id = str(uuid.uuid4())[:8]
print(f"Account: {ACCOUNT_ID}, Region: {REGION}, Unique ID: {unique_id}")

## 1. Criar um Kinesis Data Stream

Primeiro, criamos um Kinesis Data Stream na sua conta. É aqui que o AgentCore Memory publicará os eventos do ciclo de vida dos registros de memória.

Usamos um único shard (suportando até 1000 registros/seg de escrita), o que é suficiente para este tutorial. Para cargas de trabalho em produção, consulte [Resharding a Stream](https://docs.aws.amazon.com/streams/latest/dev/kinesis-using-sdk-java-resharding.html) para escalar a capacidade.

In [ ]:
stream_name = f"memory-record-stream-{unique_id}"

try:
    kinesis_client.create_stream(
        StreamName=stream_name,
        ShardCount=1  # Single shard is sufficient for this tutorial
    )
    logger.info(f"Creating Kinesis stream: {stream_name}")

    # Wait for the stream to become active
    waiter = kinesis_client.get_waiter('stream_exists')
    waiter.wait(StreamName=stream_name)

    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    stream_arn = stream_info['StreamDescription']['StreamARN']
    print(f"Kinesis stream created: {stream_arn}")

except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceInUseException':
        stream_info = kinesis_client.describe_stream(StreamName=stream_name)
        stream_arn = stream_info['StreamDescription']['StreamARN']
        print(f"Stream already exists: {stream_arn}")
    else:
        raise

## 2. Criar uma IAM Role para Streaming

O AgentCore Memory precisa de uma IAM role que ele possa assumir para publicar eventos no seu Kinesis Data Stream. Esta role requer:
- Uma **trust policy** permitindo que `bedrock-agentcore.amazonaws.com` assuma a role
- Uma **permissions policy** concedendo `kinesis:PutRecords` e `kinesis:DescribeStream` no seu stream

In [ ]:
role_name = f"AgentCoreMemoryStreamingRole-{unique_id}"

# Trust policy — allows AgentCore Memory to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# Permissions policy — scoped to our specific Kinesis stream
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "kinesis:PutRecords",
                "kinesis:DescribeStream"
            ],
            "Resource": stream_arn
        }
    ]
}

try:
    # Create the IAM role
    create_role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Allows AgentCore Memory to publish events to Kinesis"
    )
    role_arn = create_role_response['Role']['Arn']

    # Attach the inline permissions policy
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName="KinesisPublishPolicy",
        PolicyDocument=json.dumps(permissions_policy)
    )

    # Allow time for IAM propagation
    print(f"IAM role created: {role_arn}")
    print("Waiting 10 seconds for IAM propagation...")
    time.sleep(10)

except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{role_name}"
        print(f"Role already exists: {role_arn}")
    else:
        raise

## 3. Criar uma Memória com Streaming Habilitado

Agora criamos um recurso de memória do AgentCore com uma configuração de entrega de stream. Os parâmetros principais são:

- **`streamDeliveryResources`** — Aponta para nosso Kinesis stream e especifica o nível de conteúdo
- **`memoryExecutionRoleArn`** — A IAM role que o AgentCore assumirá para publicar eventos
- **`FULL_CONTENT`** — Inclui o texto do registro de memória em cada evento (use `METADATA_ONLY` para notificações mais leves)

Também configuramos uma estratégia de memória de longo prazo (preferências do usuário) para que eventos de conversação sejam extraídos em registros de memória, que por sua vez disparam eventos de streaming.

In [ ]:
memory_name = "streaming_memory"
actor_id = "demo-user"

def wait_for_memory_active(memory_id, timeout=250):
    """Poll GetMemory until the memory status is ACTIVE or timeout."""
    start = time.time()
    while time.time() - start < timeout:
        resp = agentcore_control_client.get_memory(memoryId=memory_id)
        status = resp['memory']['status']
        print(f"  Memory status: {status}")
        if status == 'ACTIVE':
            return resp['memory']
        if status == 'FAILED':
            raise RuntimeError(f"Memory creation failed: {resp['memory'].get('failureReason')}")
        time.sleep(5)
    raise TimeoutError("Timed out waiting for memory to become ACTIVE")

try:
    response = agentcore_control_client.create_memory(
        name=memory_name,
        description="Memory with long-term memory record streaming enabled",
        eventExpiryDuration=7,
        memoryExecutionRoleArn=role_arn,
        memoryStrategies=[
            {
                "userPreferenceMemoryStrategy": {
                    "name": "UserPreferences",
                    "description": "Extracts user preferences, facts, and interests from conversations",
                    "namespaces": ["/{actorId}/user_preferences/"],
                }
            }
        ],
        streamDeliveryResources={
            "resources": [
                {
                    "kinesis": {
                        "dataStreamArn": stream_arn,
                        "contentConfigurations": [
                            {
                                "type": "MEMORY_RECORDS",
                                "level": "FULL_CONTENT"
                            }
                        ]
                    }
                }
            ]
        }
    )
    memory_id = response['memory']['id']
    print(f"Memory creation initiated: {memory_id}")
    print("Waiting for memory to become ACTIVE...")
    wait_for_memory_active(memory_id)
    print(f"Memory created with streaming enabled: {memory_id}")

except ClientError as e:
    logger.error(f"Error creating memory: {e}")
    raise

## 4. Verificar se o Streaming Está Habilitado

Quando você cria uma memória com streaming, o AgentCore Memory valida a configuração e publica um evento `StreamingEnabled` no seu Kinesis stream. Vamos ler o stream para confirmar.

In [ ]:
def read_kinesis_events(stream_name, max_wait_seconds=60, max_events=10):
    """Read events from a Kinesis Data Stream.
    
    Polls the stream for new records and decodes them.
    
    Args:
        stream_name: Name of the Kinesis stream
        max_wait_seconds: Maximum time to poll before returning
        max_events: Maximum number of events to collect
    
    Returns:
        List of decoded event payloads
    """
    events = []
    
    # Get a shard iterator starting from the oldest available record
    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    shard_id = stream_info['StreamDescription']['Shards'][0]['ShardId']
    
    iterator_response = kinesis_client.get_shard_iterator(
        StreamName=stream_name,
        ShardId=shard_id,
        ShardIteratorType='TRIM_HORIZON' # to read from the oldest available record
    )
    shard_iterator = iterator_response['ShardIterator']
    
    start_time = time.time()
    while time.time() - start_time < max_wait_seconds and len(events) < max_events:
        response = kinesis_client.get_records(
            ShardIterator=shard_iterator,
            Limit=100
        )
        
        for record in response['Records']:
            data = base64.b64decode(record['Data']) if isinstance(record['Data'], str) else record['Data']
            payload = json.loads(data)
            events.append(payload)
        
        shard_iterator = response['NextShardIterator']
        
        if not response['Records']:
            time.sleep(2 Entra ID with AgentCore Gateway-pt.ipynb)
    
    return events

In [ ]:
# Check for the StreamingEnabled validation event
print("Checking for StreamingEnabled event...")
events = read_kinesis_events(stream_name, max_wait_seconds=30, max_events=1)

if events:
    for event in events:
        print(json.dumps(event, indent=2))
else:
    print("No events received yet. The StreamingEnabled event may take a moment to arrive.")

## 5. Disparar Eventos de Registros de Memória

Agora vamos gerar eventos do ciclo de vida de registros de memória criando dados de conversação. Usaremos duas abordagens:

| Abordagem | API | Como Funciona |
|:----------|:----|:--------------|
| **Opção A** | [`CreateEvent`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_CreateEvent.html) | Envia uma conversação; o AgentCore extrai **assincronamente** registros de longo prazo através da estratégia configurada |
| **Opção B** | [`BatchCreateMemoryRecords`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_BatchCreateMemoryRecords.html) | Cria registros diretamente; eventos são publicados **imediatamente** |

### Opção A: Criar eventos via memória de curto prazo (dispara extração assíncrona)

Quando você envia eventos de conversação, o AgentCore Memory extrai assincronamente registros de memória de longo prazo usando a estratégia configurada. Cada registro extraído dispara um evento `MemoryRecordCreated` no seu stream.

In [ ]:
# Send a conversation that contains extractable user preferences
agentcore_client.create_event(
    memoryId=memory_id,
    actorId=f"{actor_id}",
    sessionId="streaming-demo-session-001",
    eventTimestamp=datetime.now(timezone.utc),
    payload=[
        {
            "conversational": {
                "content": {"text": "I went hiking yesterday. I really like hiking in the Pacific Northwest. I also enjoyed the Thai restaurant. Thai food is amazing."},
                "role": "USER"
            }
        },
        {
            "conversational": {
                "content": {"text": "Great! I'll remember that you enjoy hiking in the Pacific Northwest and prefer Thai dining options."},
                "role": "ASSISTANT"
            }
        }
    ]
)

print("Conversation event sent. Long-term memory extraction will happen asynchronously.")

### Opção B: Criar registros de memória diretamente

Você também pode criar registros de memória diretamente usando a API `BatchCreateMemoryRecords`. Cada registro criado dispara imediatamente um evento `MemoryRecordCreated`.

In [ ]:
response = agentcore_client.batch_create_memory_records(
    memoryId=memory_id,
    records=[
        {
            "requestIdentifier": "direct-record-1",
            "content": {"text": "User prefers window seats on flights"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time()))
        },
        {
            "requestIdentifier": "direct-record-2",
            "content": {"text": "User's favorite programming language is Python"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time()))
        }
    ]
)

print(f"Batch created {len(response.get('successfulRecords', []))} memory records directly.")
print(f"with record IDs: \n {[i.get("memoryRecordId") for i in response.get('successfulRecords', [])]}")

## 6. Consumir Eventos de Streaming

Vamos ler o Kinesis stream para ver os eventos do ciclo de vida dos registros de memória. Como a extração de memória de longo prazo é assíncrona, faremos polling por até 90 segundos para permitir o tempo de processamento.


> **Nota para produção:** Em uma aplicação real, você poderia usar um [mapeamento de fonte de eventos do AWS Lambda](https://docs.aws.amazon.com/lambda/latest/dg/with-kinesis.html) ou a [Amazon Kinesis Client Library (KCL)](https://docs.aws.amazon.com/streams/latest/dev/shared-throughput-kcl-consumers.html) para consumir o stream em vez de fazer polling.

In [ ]:
print("Polling Kinesis stream for memory record events...\n")
events = read_kinesis_events(stream_name, max_wait_seconds=90, max_events=10)

print(f"Received {len(events)} event(s):\n")
for i, event in enumerate(events):
    stream_event = event.get("memoryStreamEvent", {})
    event_type = stream_event.get("eventType", "Unknown")
    event_time = stream_event.get("eventTime", "")
    record_id = stream_event.get("memoryRecordId", "N/A")
    record_text = stream_event.get("memoryRecordText", "")
    
    print(f"--- Event {i+1}: {event_type} ---")
    print(f"  Time:      {event_time}")
    print(f"  Memory ID: {stream_event.get('memoryId', 'N/A')}")
    print(f"  Record ID: {record_id}")
    if record_text:
        print(f"  Content:   {record_text[:120]}...")
    print()

### Inspecionar payloads completos dos eventos

Vamos examinar o JSON bruto de um evento para ver o schema completo:

In [ ]:
if events:
    # Show full payload of the first MemoryRecordCreated event
    created_events = [e for e in events if e.get("memoryStreamEvent", {}).get("eventType") == "MemoryRecordCreated"]
    if created_events:
        print("Full MemoryRecordCreated event payload:")
        print(json.dumps(created_events[0], indent=2))
    else:
        print("Full payload of first event:")
        print(json.dumps(events[0], indent=2))
else:
    print("No events to inspect. Extraction may still be in progress — try re-running this cell.")

## 7. Referência Cruzada com ListMemoryRecords

Vamos verificar se os eventos transmitidos correspondem ao que está armazenado na memória, listando os registros diretamente:

In [ ]:
records_response = agentcore_client.list_memory_records(
    memoryId=memory_id,
    namespace=f"/{actor_id}/user_preferences/"
)

records = records_response.get('memoryRecordSummaries', [])
print(f"Found {len(records)} memory record(s) in namespace '/{actor_id}/user-preferences/':\n")

for record in records:
    print(f"  Record ID: {record['memoryRecordId']}")
    print(f"  Content:   {record.get('content', {}).get('text', 'N/A')[:120]}")
    print(f"  Created:   {record.get('createdAt', 'N/A')}")
    print()

## 8. Limpeza (Opcional)

Quando terminar de experimentar, limpe os recursos criados neste tutorial:

> **Nota sobre custos:** Kinesis Data Streams incorrem em [cobranças por hora por shard](https://aws.amazon.com/kinesis/data-streams/pricing/). Certifique-se de excluir o stream quando terminar para evitar custos contínuos.

In [ ]:
# Delete the memory resource
try:
    agentcore_control_client.delete_memory(memoryId=memory_id)
    print(f"Deleting memory: {memory_id}")
    # Poll until deletion completes
    while True:
        try:
            resp = agentcore_control_client.get_memory(memoryId=memory_id)
            status = resp['memory']['status']
            print(f"  Memory status: {status}")
            if status == 'DELETING':
                time.sleep(5)
            else:
                break
        except agentcore_control_client.exceptions.ResourceNotFoundException:
            print(f"  Memory deleted successfully.")
            break
except Exception as e:
    print(f"Error deleting memory: {e}")

# Delete the Kinesis stream
try:
    kinesis_client.delete_stream(StreamName=stream_name, EnforceConsumerDeletion=True)
    print(f"Deleted Kinesis stream: {stream_name}")
except Exception as e:
    print(f"Error deleting stream: {e}")

# Delete the IAM role (must remove inline policy first)
try:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName="KinesisPublishPolicy")
    iam_client.delete_role(RoleName=role_name)
    print(f"Deleted IAM role: {role_name}")
except Exception as e:
    print(f"Error deleting IAM role: {e}")

## Conclusão

Neste tutorial, você configurou o streaming de registros de memória de ponta a ponta com o Amazon Bedrock AgentCore Memory. Você aprendeu como:

1. **Criar um Kinesis Data Stream** para receber eventos do ciclo de vida de registros de memória
2. **Configurar uma IAM role** com permissões de menor privilégio para o AgentCore publicar no seu stream
3. **Criar um recurso de memória** com streaming habilitado e entrega `FULL_CONTENT`
4. **Disparar eventos** tanto por extração de conversação quanto por criação direta de registros
5. **Consumir e inspecionar** eventos `MemoryRecordCreated` do stream em tempo real

### Próximos Passos
Para aprender mais sobre como usar a capacidade de streaming do AgentCore Memory de forma mais eficiente, você pode tentar o seguinte:
- **Adicionar um consumidor Lambda** para processar eventos automaticamente (veja a [documentação](https://docs.aws.amazon.com/bedrock-agentcore/latest/userguide/memory-streaming.html) para um exemplo)
- **Mudar para `METADATA_ONLY`** no nível de conteúdo para reduzir a transferência de dados quando você precisa apenas de notificações de alteração
- **Configurar alarmes do CloudWatch** nas métricas `StreamPublishingFailure` e `StreamUserError` para monitoramento em produção
- **Construir workflows orientados a eventos** — sincronizar registros de memória com um data lake no S3, disparar notificações ou atualizar perfis de usuários downstream